In [6]:
import os
import pandas as pd
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output
import requests


# LOAD THE DATASET
interactions_df = pd.read_csv("data/buffalo_gals_community_v1.csv").fillna("")

scenes = [
    {
        "character": "Chickadee",
        "id": "COM038",
        "question": "Will you help Chickadee prepare dinner?",
        "choice": "Help cook",
        "points": 3,
        "image": "chickadee.png",
        "action_image": "chickadeeMyraCookTogether.png"
    },
    {
        "character": "Coyote",
        "id": "COM002",
        "question": "Will you follow Coyote down to the creek?",
        "choice": "Follow Coyote",
        "points": 1,
        "image": "coyote.png",
        "action_image": "theCreek.png"
    },
    {
        "character": "Grandmother",
        "id": "COM053",
        "question": "Will you listen to Grandmother's guidance?",
        "choice": "Listen",
        "points": 2,
        "image": "grandmother.png",
        "action_image": "theCave.png"
    }
]




# FIND EXACTLY ONE RECORD: never invent a missing outcome.
def get_interaction(interaction_id):
    match = interactions_df[
        interactions_df["interaction_id"] == interaction_id
    ]
    if len(match) != 1:
        raise ValueError(f"Expected one row for {interaction_id}; found {len(match)}.")
    return match.iloc[0]

# Check the selected records before displaying the game.
for scene in scenes:
    row = get_interaction(scene["id"])
    participants = f"{row['actor']}; {row['recipient']}".split(";")
    if "Myra" not in [name.strip() for name in participants]:
        raise ValueError(f"Myra is not listed as a participant in {scene['id']}.")

# DISPLAY AN ASSIGNED IMAGE; missing artwork does not stop the game.
def show_image(image_name, size=450):
    if image_name is None:
        return
    path = f"images/{image_name}"
    try:
        with Image.open(path) as img:
            img.thumbnail((size, size), Image.Resampling.NEAREST)
            display(img)
    except OSError:
        print(f"[Image unavailable: {path}]")




# GAME STATE
community_points = 0
completed_ids = []
current_scene = None
phase = "menu"
max_points = sum(scene["points"] for scene in scenes)

# WIDGETS
output = widgets.Output()
score = widgets.Label()
menu_buttons = []
for scene in scenes:
    menu_buttons.append(widgets.Button(layout=widgets.Layout(width="350px")))

finish_button = widgets.Button(description="Finish exploring")
menu = widgets.VBox(menu_buttons + [finish_button])
action_button = widgets.Button(layout=widgets.Layout(width="220px"))
leave_button = widgets.Button(description="Not now")
choices = widgets.HBox([action_button, leave_button])
explore_button = widgets.Button(description="Let's explore!")
restart_button = widgets.Button(description="Play again")


def update_score():
    score.value = f"Player: Myra | Community Points: {community_points} / {max_points}"


# HOME
def show_menu(b=None):
    global current_scene, phase
    current_scene = None
    phase = "menu"

    for button, scene in zip(menu_buttons, scenes):
        done = scene["id"] in completed_ids
        button.disabled = done
        if done:
            button.description = f"{scene['character']}: completed (+{scene['points']})"
        else:
            button.description = f"{scene['character']}: {scene['choice']} (+{scene['points']})"

    with output:
        clear_output(wait=True)
        print("HI MYRA! \nNOW THAT YOU ARE IN THE DESERT, YOU WILL NEED TO CONNECT WITH THE ANIMAL COMMUNITTY IN ORDER TO SURVIVE. \nCOMMUNITY POINTS WILL GIVE YOU WHAT YOU NEED TO KEEP GOING. \nWHERE WILL YOU GO?")
        print("Choose someone to meet. You can explore in any order.")
        if len(completed_ids) == len(scenes):
            print("\nAll three activities are complete! Finish exploring to see your total.")

    menu.layout.display = "flex"
    choices.layout.display = "none"
    explore_button.layout.display = "none"
    restart_button.layout.display = "none"


# MEET: show choices only after choosing a character.
def on_meet_click(b):
    global current_scene, phase
    if phase != "menu" or b.disabled:
        return

    current_scene = scenes[menu_buttons.index(b)]
    interaction = get_interaction(current_scene["id"])
    phase = "encounter"

    with output:
        clear_output(wait=True)
        print(f"Location: {interaction['location']}")
        show_image(current_scene["image"], 350)
        #print(f"\nYou are Myra, meeting {current_scene['character']}.")
        print(current_scene["question"])

    action_button.description = f"{current_scene['choice']} (+{current_scene['points']})"
    menu.layout.display = "none"
    choices.layout.display = "flex"

    

# ACT: descriptions and outcomes come from the CSV
def on_action_click(b):
    global community_points, phase
    if phase != "encounter":
        return
    if current_scene["id"] in completed_ids:
        return

    interaction = get_interaction(current_scene["id"])
    phase = "result"
    completed_ids.append(current_scene["id"])
    community_points += current_scene["points"]
    update_score()

    with output:
        clear_output(wait=True)
        # Reuse the character portrait when no action image is assigned.
        show_image(current_scene["action_image"] or current_scene["image"], 500)
        print(interaction["description"])
        if interaction["outcome"]:
            print("\nOUTCOME:")
            print(interaction["outcome"])
        print(f"\n+{current_scene['points']} Community Points!")
        print(f"Total so far: {community_points} / {max_points}")
        #print(f"\nSource: {interaction['interaction_id']} | PDF page {interaction['pdf_page']}")

    choices.layout.display = "none"
    explore_button.layout.display = ""


# DECLINE: this option remains available on the menu for a later visit.
def on_leave_click(b):
    global phase
    if phase != "encounter":
        return
    phase = "result"
    with output:
        clear_output(wait=True)
        print("You decide not to take part right now. You can come back later.")
        print("No points added.")
    choices.layout.display = "none"
    explore_button.layout.display = ""


# END: show the points actually earned, including partial playthroughs.
def on_finish_click(b):
    global phase
    if phase != "menu":
        return
    phase = "end"
    with output:
        clear_output(wait=True)
        print("EXPLORATION COMPLETE")
        print(f"\nYou earned {community_points} of {max_points} Community Points!")
        print(f"Activities completed: {len(completed_ids)} / {len(scenes)}\n")
        for scene in scenes:
            earned = scene["points"] if scene["id"] in completed_ids else 0
            print(f"{scene['character']} - {scene['choice']}: +{earned}")

    menu.layout.display = "none"
    choices.layout.display = "none"
    explore_button.layout.display = "none"
    restart_button.layout.display = ""


def on_restart_click(b):
    global community_points, completed_ids
    community_points = 0
    completed_ids = []
    update_score()
    show_menu()



# CONNECT BUTTONS
for button in menu_buttons:
    button.on_click(on_meet_click)

action_button.on_click(on_action_click)
leave_button.on_click(on_leave_click)
explore_button.on_click(show_menu)
finish_button.on_click(on_finish_click)
restart_button.on_click(on_restart_click)

# DISPLAY ONE GAME SCREEN
if "game_screen" in globals():
    game_screen.close()

game_screen = widgets.VBox([
    score, output, menu, choices, explore_button, restart_button
])
update_score()
show_menu()
from IPython.display import Image as NotebookImage, display

with output:
    display(NotebookImage(filename="images/myra.png", width=150))

display(game_screen)
